# Flash Floods KY Python Scripts

## Script #1: Moon & Sun Calculations

In [ ]:
# Import Libraries and Dependencies
from datetime import datetime
from pathlib import Path
import math
import ephem
import pandas as pd

In [ ]:
# Read dataset into dataframe, then inspect it
df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

df.head()

In [ ]:
# Create a function that calculates sunrise or sunset for an observer
def get_sun_event(observer, sun, event, never_message):
    """Calculate sunrise or sunset for an observer."""
    
    try:
        result = event(sun)
        return result.datetime().strftime("%Y-%m-%d %H:%M:%S")
    
    except (ephem.CircumpolarError, ephem.AlwaysUpError):
        return "Sun always up"
    
    except ephem.NeverUpError:
        return never_message

In [ ]:
# Create a function that determines the moon's phase from illumination and lunar age
def determine_moon_phase(illumination, lunation_age):
    """Determine the moon's phase from illumination and lunar age."""
    
    if illumination < 0.03:
        return "New Moon"
    
    if illumination > 0.97:
        return "Full Moon"
    
    if lunation_age < 14.77:
        if illumination < 0.45:
            return "Waxing Crescent"
        if illumination < 0.55:
            return "First Quarter"
        return "Waxing Gibbous"
    
    if illumination > 0.55:
        return "Waning Gibbous"
    
    if illumination > 0.45:
        return "Third Quarter"
    
    return "Waning Crescent"

In [ ]:
# Create a function that calculates local sun and moon data for one row before applying it to the entire dataframe
def calculate_local_astro_data(row: pd.Series) -> pd.Series:
    """Calculate local sun and moon data for one row."""
    
    try:
        # Get event date
        date_obj = datetime.strptime(
            row["begin_date"],
            "%Y-%m-%d"
        )
        
        # Create observer using event coordinates
        observer = ephem.Observer()
        observer.lat = str(row["begin_lat"])
        observer.lon = str(row["begin_lon"])
        observer.date = date_obj
        
        # --------------------------------------------------
        # SUN CALCULATIONS
        # --------------------------------------------------
        
        sun = ephem.Sun()
        sun.compute(observer)
        
        sun_altitude = math.degrees(sun.alt)
        sun_azimuth = math.degrees(sun.az)
        
        # Calculate sunrise and sunset
        observer.date = date_obj.date()
        
        sunrise = get_sun_event(
            observer,
            sun,
            observer.next_rising,
            "Sun never rises"
        )
        
        sunset = get_sun_event(
            observer,
            sun,
            observer.next_setting,
            "Sun never sets"
        )
        
        # --------------------------------------------------
        # MOON CALCULATIONS
        # --------------------------------------------------
        
        observer.date = date_obj
        
        moon = ephem.Moon()
        moon.compute(observer)
        
        moon_altitude = math.degrees(moon.alt)
        moon_azimuth = math.degrees(moon.az)
        
        # Calculate moon illumination
        illumination = moon.phase / 100
        
        # Calculate lunar age
        previous_new_moon = ephem.previous_new_moon(observer.date)
        next_new_moon = ephem.next_new_moon(observer.date)
        
        lunation_age = (
            (observer.date - previous_new_moon)
            * 29.53
            / (next_new_moon - previous_new_moon)
        )
        
        # Determine moon phase
        moon_phase = determine_moon_phase(
            illumination,
            lunation_age
        )
        
        # Return results
        return pd.Series({
            "sun_altitude_deg": round(sun_altitude, 2),
            "sun_azimuth_deg": round(sun_azimuth, 2),
            "sunrise_utc": sunrise,
            "sunset_utc": sunset,
            "moon_altitude_deg": round(moon_altitude, 2),
            "moon_azimuth_deg": round(moon_azimuth, 2),
            "moon_phase_name": moon_phase,
            "moon_illumination_pct": round(illumination * 100, 2)
        })
        
    except Exception as e:
        return pd.Series({
            "sun_altitude_deg": None,
            "sun_azimuth_deg": None,
            "sunrise_utc": f"Error: {e}",
            "sunset_utc": f"Error: {e}",
            "moon_altitude_deg": None,
            "moon_azimuth_deg": None,
            "moon_phase_name": f"Error: {e}",
            "moon_illumination_pct": None
        })

In [ ]:
# Test calculate_local_astro_data function with the first row of the dataframe
test_result = calculate_local_astro_data(df.iloc[0])

test_result

In [ ]:
# Run astronomical calculations and display results
astro_data = df.apply(calculate_local_astro_data, axis=1)

astro_data.head()

In [ ]:
# Combine event_id with the calculated astronomy data
output_df = pd.concat(
    [df[["event_id"]], astro_data],
    axis=1
)

output_df.tail()

In [ ]:
# Save dataframe as CSV file
output_df.to_csv("../data/processed/flash_floods_ky_moon_sun_data.csv", index=False)

## Script #2: Chi-Square Goodness of Fit

In [ ]:
# Import Libraries and Dependencies
from pathlib import Path
import pandas as pd
from scipy.stats import chisquare

In [ ]:
# Read dataset into dataframe
df = pd.read_csv("../data/processed/flash_floods_ky_moon_sun_data.csv")

df.head()

In [ ]:
# Create bin boundaries 
bins = [0, 10, 20, 30, 40, 50, 60, 70, 80, 90, 100]

In [ ]:
# Create bin labels
labels = [
    "0-10%", 
    "10-20%", 
    "20-30%", 
    "30-40%", 
    "40-50%", 
    "50-60%", 
    "60-70%", 
    "70-80%", 
    "80-90%", 
    "90-100%"
]

In [ ]:
# Assign each flash flooding event to a moon illumination bin
df["illumination_bin"] = pd.cut(
    df["moon_illumination_pct"], 
    bins=bins, 
    labels=labels, 
    include_lowest=True
)

df[["moon_illumination_pct", "illumination_bin"]].head(20)

In [ ]:
# Count observed flash flooding events
observed = df["illumination_bin"].value_counts().sort_index()

observed

In [ ]:
# Calculate expected flash flooding event counts
expected = [len(df) / len(observed)] * len(observed)

expected

In [ ]:
# Run Chi-Square Goodness of Fit test and display the results
chi2, p_value = chisquare(
    f_obs=observed, 
    f_exp=expected
)

print("Observed Counts:")
print(observed)

print("\nExpected Counts:")
print(expected)

print(f"\nChi-Square Statistic: {chi2:.4f}")
print(f"P-Value: {p_value:.6f}")

## Script #3: Oceanic Nino Index 

In [ ]:
# Import Libraries and Dependencies
import io
from pathlib import Path
import numpy as np
import pandas as pd

In [ ]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

In [ ]:
# Read the ONI text file
with open("../data/raw/oni_backup.txt", "r", encoding="utf-8") as f: oni_text = f.read()

print(oni_text[:1000])

In [ ]:
# Parse ONI text into a dataframe, then check results
oni_df = pd.read_csv(
    io.StringIO(oni_text.strip()),
    sep=r"\s+",
    dtype={"YR": int, "ANOM": float}
)

oni_df.head()

In [ ]:
# Map ONI seasons to a central calendar month, then check results
season_to_month = {
    "DJF": 1,
    "JFM": 2,
    "FMA": 3,
    "MAM": 4,
    "AMJ": 5,
    "MJJ": 6,
    "JJA": 7,
    "JAS": 8,
    "ASO": 9,
    "SON": 10,
    "OND": 11,
    "NDJ": 12,
}

oni_df["month"] = oni_df["SEAS"].map(season_to_month)

oni_df.head()

In [ ]:
# Create a date column using the year and month, then check results
oni_df["date"] = pd.to_datetime(
    dict(
        year=oni_df["YR"],
        month=oni_df["month"],
        day=1
    )
)

oni_df.head()

In [ ]:
# Classify ENSO phase based on ONI anomaly, then check results
oni_df["enso_phase"] = np.select(
    [
        oni_df["ANOM"] >= 0.5,
        oni_df["ANOM"] <= -0.5
    ],
    [
        "El Nino",
        "La Nina"
    ],
    default="Neutral"
)

oni_df.head()

In [ ]:
# Select the columns needed for the analysis and rename them for clarity, then check results
oni_df = oni_df[
    ["date", "SEAS", "ANOM", "enso_phase"]
].rename(
    columns={
        "SEAS": "oni_season",
        "ANOM": "oni_anomaly"
    }
)

oni_df.head()

In [ ]:
# Sort ONI data chronologically, then check results
oni_df = oni_df.sort_values("date").reset_index(drop=True)

oni_df.head()

In [ ]:
# Sort flash flood data chronologically, then check results
flood_df = flood_df.sort_values("begin_date").reset_index(drop=True)

flood_df[["event_id", "begin_date"]].head()

In [ ]:
# Check the data types of the columns before merging into one dataframe
print("flood_df begin_date:", flood_df["begin_date"].dtype)
print("oni_df date:", oni_df["date"].dtype)

In [ ]:
# Change flood_df begin_date from str datatype to datetime datatype
flood_df["begin_date"] = pd.to_datetime(
    flood_df["begin_date"]
)

print(flood_df["begin_date"].dtype)

In [ ]:
# Merge each flash flood event with the most recent ONI record occurring on or before the event date, then check results
enriched_df = pd.merge_asof(
    flood_df,
    oni_df,
    left_on="begin_date",
    right_on="date",
    direction="backward"
)

enriched_df.head()

In [ ]:
# Keep event_id and the ONI columns
enriched_df = enriched_df[
    [
        "event_id",
        "oni_season",
        "oni_anomaly",
        "enso_phase"
    ]
]

enriched_df.head()

In [ ]:
# Save the merged DataFrame as a CSV file
enriched_df.to_csv("../data/processed/flash_floods_ky_oni_data.csv", index=False)

## Script #4: NLCD Landcover API

In [ ]:
# Import Libraries and Dependencies 
import os
from pathlib import Path
import pandas as pd
from arcgis.gis import GIS
from arcgis.raster import ImageryLayer
from concurrent.futures import ThreadPoolExecutor, as_completed
from dotenv import load_dotenv

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Add configuration
API_KEY = os.environ.get("arcgis_api_key")
ARCGIS_URL = "https://arcgis.com"

In [ ]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_event_info.csv")

flood_df.head()

In [ ]:
# Define NLCD Layer IDs
LAYER_IDS = {
    "nlcd":"32e2ccc6416746a9a72b4d216813f84f",
    "elev":"58a541efc59545e6b7137f961d7de883",
    "imperv":"6df535f263dd44f489365eed49461a38",
}

In [ ]:
# Define NLCD classes
NLCD_CLASSES = {
    11: "Open Water",
    21: "Developed, Open Space",
    22: "Developed, Low Intensity",
    23: "Developed, Medium Intensity",
    24: "Developed, High Intensity",
    31: "Barren Land",
    41: "Deciduous Forest",
    42: "Evergreen Forest",
    43: "Mixed Forest",
    52: "Shrub/Scrub",
    71: "Grassland/Herbaceous",
    81: "Pasture/Hay",
    82: "Cultivated Crops",
    90: "Woody Wetlands",
    95: "Emergent Herbaceous Wetlands",
}

In [ ]:
# Connect to ArcGIS
def initialize_layers(api_key: str) -> tuple:
    """Connects to the ArcGIS GIS API with strict SSL validation
    and returns the three imagery layers."""

    print("Connecting to ArcGIS Living Atlas layers...")

    gis = GIS(
        ARCGIS_URL,
        api_key=api_key,
        verify_cert=True
    )

    layers = {
        name: ImageryLayer(
            gis.content.get(item_id).url,
            gis=gis
        )
        for name, item_id in LAYER_IDS.items()
    }

    print("All layers connected successfully.")

    return (
        layers["nlcd"],
        layers["elev"],
        layers["imperv"]
    )

In [ ]:
# Query One Imagery Layer
def query_layer_value(layer: ImageryLayer, geom: dict):
    """Queries an imagery layer at a point geometry
    and returns its pixel value, or None on failure."""

    try:
        response = layer.identify(
            geometry=geom,
            return_pixel_values=True
        )

        return response.get("value", None)

    except Exception:
        return None

In [ ]:
# Process One Flash Flood Event
def process_single_row(idx, row, layers):
    """Worker function to process a single row's
    spatial variables concurrently."""

    nlcd_layer, elev_layer, imp_layer = layers

    mid_lat = (row["begin_lat"] + row["end_lat"]) / 2
    mid_lon = (row["begin_lon"] + row["end_lon"]) / 2

    if pd.isna(mid_lat) or pd.isna(mid_lon):
        return idx, (None, None, None, None)

    geom = {
        "x": mid_lon,
        "y": mid_lat,
        "spatialReference": {"wkid": 4326}
    }

    val_nlcd = query_layer_value(nlcd_layer, geom)
    val_elev = query_layer_value(elev_layer, geom)
    val_imp = query_layer_value(imp_layer, geom)

    c_code = int(val_nlcd) if val_nlcd is not None else None

    c_class = (
        NLCD_CLASSES.get(c_code, "Unknown")
        if c_code is not None
        else None
    )

    elevation = (
        float(val_elev)
        if val_elev is not None
        else None
    )

    impervious = (
        int(val_imp)
        if val_imp is not None
        else None
    )

    return idx, (
        c_code,
        c_class,
        elevation,
        impervious
    )

In [ ]:
# Process all flash flooding events within the dataframe
def fetch_geospatial_attributes(
    df: pd.DataFrame,
    layers: tuple,
    max_workers: int = 20,
) -> pd.DataFrame:

    """Fetches geospatial attributes for each row
    and joins them onto the dataframe."""

    print(
        f"Starting batch execution using "
        f"{max_workers} concurrent threads..."
    )

    cols = [
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]

    results = {}

    total_rows = len(df)

    with ThreadPoolExecutor(
        max_workers=max_workers
    ) as executor:

        futures = {
            executor.submit(
                process_single_row,
                idx,
                row,
                layers
            ): idx

            for idx, row in df.iterrows()
        }

        for completed, future in enumerate(
            as_completed(futures), 1
        ):

            idx, values = future.result()

            results[idx] = values

            if completed % 100 == 0 or completed == total_rows:
                print(
                    f"Progress: {completed}/{total_rows} "
                    f"rows extracted "
                    f"({completed/total_rows*100:.1f}%)"
                )

    result_df = pd.DataFrame.from_dict(
        results,
        orient="index",
        columns=cols
    )

    return df.join(result_df)

In [ ]:
# Connect to ArcGIS
atlas_layers = initialize_layers(API_KEY)

In [ ]:
# Run geospatial extraction
processed_df = fetch_geospatial_attributes(
    flood_df,
    atlas_layers,
    max_workers=25
)

In [ ]:
# Inspect processed dataset
processed_df.head()

In [ ]:
# Keep only event_id and NLCD columns
processed_df = processed_df[
    [
        "event_id",
        "nlcd_code",
        "nlcd_class",
        "elevation_m",
        "impervious_surface_pct"
    ]
]

processed_df.head()

In [ ]:
# Save processed dataset as CSV file
processed_df.to_csv("../data/processed/flash_floods_ky_nlcd_data.csv", index=False)

## Script #5: Weather API

In [ ]:
# Import Libraries and Dependencies
import pandas as pd
import requests
import json
import time
from dotenv import load_dotenv
import os

In [ ]:
# Load environment variables
load_dotenv()

In [ ]:
# Get the weather API key from environment variables
weather_api_key = os.getenv("weather_api_key")

In [ ]:
# Read in dataset as dataframe
flood_df = pd.read_csv("../data/processed/flash_floods_ky_2015_2025_cleaned.csv")

flood_df.head()

In [ ]:
# Prepare a dataframe for weather extraction
weather_tasks = df[
    [
        "event_id",
        "begin_date",
        "begin_lat",
        "begin_lon",
        "begin_location"
    ]
].copy()

weather_tasks = weather_tasks.dropna(
    subset=[
        "event_id",
        "begin_date",
        "begin_lat",
        "begin_lon"
    ]
).copy()

print(f"Records ready for weather extraction: {len(weather_tasks)}")

In [ ]:
# Define weather fields to extract
WEATHER_FIELDS = [
    "maxtemp_f",
    "mintemp_f",
    "avgtemp_f",
    "maxwind_mph",
    "totalprecip_in",
    "avgvis_miles",
    "avghumidity",
    "uv",
    "daily_will_it_rain",
    "daily_chance_of_rain",
    "condition_text"
]

In [ ]:
# Create a function to get historical weather data from WeatherAPI
def get_historical_weather(
    event_id,
    event_date,
    latitude,
    longitude,
    api_key,
    max_retries=3
):
    
    url = "https://api.weatherapi.com/v1/history.json"
    
    params = {
        "key": api_key,
        "q": f"{latitude},{longitude}",
        "dt": event_date
    }
    
    for attempt in range(max_retries):
        
        try:
            response = requests.get(
                url,
                params=params,
                timeout=60
            )
            
            response.raise_for_status()
            
            data = response.json()
            
            day_data = data["forecast"]["forecastday"][0]["day"]
            location_data = data["location"]
            condition_data = day_data.get("condition", {})
            
            result = {
                "event_id": event_id,
                "date": event_date,
                "latitude": latitude,
                "longitude": longitude,
                "location": location_data.get("name")
            }
            
            # Extract standard weather fields
            for field in WEATHER_FIELDS:
                
                if field not in [
                    "condition_text"
                ]:
                    result[field] = day_data.get(field)
            
            # Extract nested condition fields
            result["condition_text"] = condition_data.get("text")
            
            return result
            
        except requests.exceptions.RequestException as e:
            
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
            
            else:
                return {
                    "event_id": event_id,
                    "date": event_date,
                    "latitude": latitude,
                    "longitude": longitude,
                    "location": None,
                    "error": str(e)
                }

In [ ]:
# Test the get_historical_weather function with the first row of the weather_tasks dataframe
test_row = weather_tasks.iloc[0]

test_result = get_historical_weather(
    event_id=test_row["event_id"],
    event_date=pd.to_datetime(
        test_row["begin_date"]
    ).strftime("%Y-%m-%d"),
    latitude=test_row["begin_lat"],
    longitude=test_row["begin_lon"],
    api_key=weather_api_key
)

test_result

In [ ]:
# Run weather extraction for all records in the weather_tasks dataframe
weather_results = []

total_records = len(weather_tasks)

print("Starting weather data extraction...")
print(f"Total records: {total_records}")

for i, (_, row) in enumerate(weather_tasks.iterrows(), start=1):
    
    result = get_historical_weather(
        event_id=row["event_id"],
        event_date=pd.to_datetime(
            row["begin_date"]
        ).strftime("%Y-%m-%d"),
        latitude=row["begin_lat"],
        longitude=row["begin_lon"],
        api_key=weather_api_key
    )
    
    weather_results.append(result)
    
    if i % 25 == 0 or i == total_records:
        print(
            f"Processed {i:,} / {total_records:,} "
            f"({i / total_records:.1%})"
        )
    
    # Pause between requests
    time.sleep(0.2)

print("Weather extraction complete!")

In [ ]:
# Create weather dataframe
weather_df = pd.DataFrame(weather_results)

weather_df.head()

In [ ]:
# Drop unnecessary columns and display the first few rows of the weather dataframe
weather_df = weather_df.drop(columns=
    ['date',
    'latitude',
    'longitude',
    'location'
    ])

weather_df.head()

In [ ]:
# Save dataframe as CSV
weather_df.to_csv("../data/raw/flash_floods_ky_weather_conditions.csv",index=False)